# core

> Fill in a module description here

In [1]:
# | default_exp db

In [2]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [3]:
# | export
from typing import (
    Any,
    List,
    Sequence,
    Optional,
    cast,
    Type,
    TypeVar,
    Union,
    Literal,
    Tuple,
    Callable,
    Protocol,
    ParamSpec,
    Concatenate,
    Generic,
)
from functools import wraps
from enum import Enum, IntEnum
from typing_extensions import Annotated
from abc import ABC, abstractmethod
from sqlalchemy import BigInteger, text, LargeBinary
from sqlalchemy.orm import selectinload, joinedload
from pydantic.functional_validators import BeforeValidator
from sqlmodel import Field, Session, SQLModel, create_engine, select, col, Relationship
from datetime import datetime
from uuid import uuid4, UUID
from sqlalchemy import Column, JSON, case
from passlib.hash import pbkdf2_sha256
import re
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# |export


def validate_file_path(v: str) -> str:
    """Check that the path is valid according to regex"""
    if not re.match(r"^(\/[a-zA-Z0-9_-]+)+\/[a-zA-Z0-9_-]+\.[a-zA-Z0-9_-]+$", v):
        raise ValueError(f"Path {v} is not valid")
    return v

In [5]:
test_path = validate_file_path("/Users/max/Documents/product_horse/nbs/test.txt")
print(test_path)
try:
    not_path_a = validate_file_path("/not/a/valid/path/")
except ValueError:
    print("correct - doesn't work!")
else:
    raise Exception("Shouldn't pass!")
try:
    not_path_b = validate_file_path("_sdf-sdfsdfs/")
except ValueError:
    print("correct - doesn't work!")
else:
    raise Exception("Shouldn't pass!")

/Users/max/Documents/product_horse/nbs/test.txt
correct - doesn't work!
correct - doesn't work!


In [6]:
# | export

STRINGS_TO_SANITIZE = ["\x00"]


def sanitize_strings(v: str) -> str:
    for string in STRINGS_TO_SANITIZE:
        v = v.replace(string, "")
    return v


# https://docs.pydantic.dev/latest/concepts/validators/#annotated-validators
ValidString = Annotated[str, BeforeValidator(sanitize_strings)]
ValidPath = Annotated[ValidString, BeforeValidator(validate_file_path)]

In [7]:
# | export

### AUTH MODELS ###


class VideoVisibility(str, Enum):
    PRIVATE = "private"
    INTERNAL = "internal"
    PUBLIC = "public"


class PermissionLevel(IntEnum):
    PUBLIC = 0
    READ = 1
    WRITE = 2
    ADMIN = 3


class UnvalidatedCompany(SQLModel):
    name: ValidString = Field(default=None)


class Company(UnvalidatedCompany, table=True):
    id: UUID = Field(default_factory=uuid4, primary_key=True)
    created_at: datetime = Field(default=datetime.utcnow(), nullable=False)
    updated_at: datetime = Field(default_factory=datetime.utcnow, nullable=False)
    employees: list["Employee"] = Relationship(back_populates="company")


class UnvalidatedEmployee(SQLModel):
    email: ValidString = Field(default=None)
    name: ValidString = Field(default=None)
    permission_level: PermissionLevel = Field(default=PermissionLevel.READ)


class CreateEmployee(UnvalidatedEmployee):
    password: str


class Employee(UnvalidatedEmployee, table=True):
    id: UUID = Field(default_factory=uuid4, primary_key=True)
    created_at: datetime = Field(default=datetime.utcnow(), nullable=False)
    updated_at: datetime = Field(default_factory=datetime.utcnow, nullable=False)
    company: Company = Relationship(back_populates="employees")
    company_id: UUID = Field(foreign_key="company.id")
    hashed_password: bytes = Field(sa_column=Column(LargeBinary), default=None)


class OrgBoundModel(SQLModel):
    company_id: UUID = Field(foreign_key="company.id")
    created_by_id: UUID = Field(foreign_key="employee.id")

Below, all fields that are raw user input should go on UnvalidatedModel for validation

In [8]:
# |export


class UnvalidatedUser(OrgBoundModel):
    name: ValidString = Field(default=None)
    external_id: ValidString = Field(default=None)


class User(UnvalidatedUser, table=True):
    """This represents the COMPANY's users, not the users of producthorse"""

    id: UUID = Field(default_factory=uuid4, primary_key=True)
    created_at: datetime = Field(default=datetime.utcnow(), nullable=False)
    updated_at: datetime = Field(default_factory=datetime.utcnow, nullable=False)
    file_metadata: List["FileMetadata"] = Relationship(
        back_populates="user", sa_relationship_kwargs={"cascade": "all, delete-orphan"}
    )


class UnvalidatedFileMetadata(OrgBoundModel):
    user_id: UUID = Field(default=None, foreign_key="user.id")
    file_name: ValidString = Field(default=None)
    file_path: ValidString = Field(default=None)


class FileMetadata(UnvalidatedFileMetadata, table=True):
    __tablename__ = "file_metadata"
    id: UUID = Field(default_factory=uuid4, primary_key=True)
    created_at: datetime = Field(default=datetime.utcnow(), nullable=False)
    updated_at: datetime = Field(default_factory=datetime.utcnow, nullable=False)
    transcription: "Transcription" = Relationship(
        back_populates="file_metadata",
        sa_relationship_kwargs={"cascade": "all, delete-orphan"},
    )
    user: "User" = Relationship(back_populates="file_metadata")


class UnvalidatedWord(OrgBoundModel):
    confidence: float
    end: int = Field(sa_column=Column(BigInteger))
    speaker: ValidString = Field(default=None)
    start: int = Field(sa_column=Column(BigInteger))
    text: ValidString = Field(default=None)


class WordClipLink(SQLModel, table=True):
    word_id: UUID = Field(default=None, foreign_key="word.id", primary_key=True)
    clip_id: UUID = Field(default=None, foreign_key="clip.id", primary_key=True)


class Word(UnvalidatedWord, table=True):
    id: UUID = Field(default_factory=uuid4, primary_key=True)
    utterance_id: UUID = Field(default=None, foreign_key="utterance.id")
    utterance: "Utterance" = Relationship(back_populates="words")
    clips: List["Clip"] = Relationship(back_populates="words", link_model=WordClipLink)


# Base class -- do not use directly
class UtteranceBase(OrgBoundModel):
    confidence: float
    end: int = Field(sa_column=Column(BigInteger))
    speaker: ValidString = Field(default=None)
    start: int = Field(sa_column=Column(BigInteger))
    text: ValidString = Field(default=None)


class UtteranceSegment(
    UtteranceBase
):  # mocking the Utterance without actually needing db connects
    """
    This is a mock class for creating utterance segments from a transcription.
    It is not a database model and does not have a corresponding table.
    """

    id: UUID | None = Field(default=None)
    transcription_id: UUID = Field(default=None)
    transcription: "Transcription"
    words: list["Word"]
    custom_start: float | None = None
    custom_end: float | None = None


class UnvalidatedUtterance(UtteranceBase):
    words: list["Word"] | list["UnvalidatedWord"]


class Utterance(UtteranceBase, table=True):
    id: UUID = Field(default_factory=uuid4, primary_key=True)
    transcription: "Transcription" = Relationship(back_populates="utterances")
    transcription_id: UUID = Field(default=None, foreign_key="transcription.id")
    words: list["Word"] = Relationship(
        back_populates="utterance",
        sa_relationship_kwargs={"cascade": "all, delete-orphan"},
    )
    # relationship field lengths aren't validatable in sqlmodel :-/
    # PydanticUserError: Decorators defined with incorrect fields: __main__.Utterance:4797847184.check_words_length (use check_fields=False if you're inheriting from the model and intended this)
    # @field_validator('words')
    # def check_words_length(cls, values: list["Word"]) -> list["Word"]:
    #     if not values:
    #         raise ValueError("Utterance must have at least one word.")
    #     return values


class UnvalidatedTranscription(OrgBoundModel):
    file_id: UUID = Field(default=None, foreign_key="file_metadata.id")
    text: ValidString = Field(default=None)


class Transcription(UnvalidatedTranscription, table=True):
    id: UUID = Field(default_factory=uuid4, primary_key=True)
    created_at: datetime = Field(default=datetime.utcnow(), nullable=False)
    updated_at: datetime = Field(default_factory=datetime.utcnow, nullable=False)
    embedded: Optional[bool] = Field(default=False)
    utterances: list["Utterance"] = Relationship(
        back_populates="transcription",
        sa_relationship_kwargs={"cascade": "all, delete-orphan"},
    )
    file_metadata: "FileMetadata" = Relationship(back_populates="transcription")

    def __hash__(self):
        return hash(self.id)


class UnvalidatedSchema(OrgBoundModel):
    input_text: ValidString = Field(default=None)
    json_schema: dict[str, Any] = Field(default={}, sa_column=Column(JSON))


class Schema(UnvalidatedSchema, table=True):
    id: UUID = Field(default_factory=uuid4, primary_key=True)
    created_at: datetime = Field(default=datetime.utcnow(), nullable=False)
    updated_at: datetime = Field(default_factory=datetime.utcnow, nullable=False)


class UnvalidatedVideo(OrgBoundModel):
    duration: float
    title: ValidString = Field(default=None)
    file_path: str | None = Field(
        default=None
    )  # would like to use better validation here, but there seems to be a bug in sqlmodel https://github.com/tiangolo/sqlmodel/issues/67
    resolution_x: int | None = Field(default=None)
    resolution_y: int | None = Field(default=None)
    fps: int | None = Field(default=None)


class RenderStatus(str, Enum):
    pending = "pending"
    processing = "processing"
    complete = "complete"
    failed = "failed"


class Video(UnvalidatedVideo, table=True):
    id: UUID = Field(default_factory=uuid4, primary_key=True)
    created_at: datetime = Field(default=datetime.utcnow(), nullable=False)
    updated_at: datetime = Field(default_factory=datetime.utcnow, nullable=False)
    clips: List["Clip"] = Relationship(
        back_populates="video", sa_relationship_kwargs={"cascade": "all, delete-orphan"}
    )
    render_status: RenderStatus = Field(default=RenderStatus.pending)
    visibility: VideoVisibility = Field(default=VideoVisibility.PRIVATE)


class VideoType(str, Enum):
    video = "video"
    audio = "audio"


class UnvalidatedClip(OrgBoundModel):
    utterance_start: float
    utterance_end: float
    duration: float
    fps: int
    resolution_x: int
    resolution_y: int
    speaker_role: Optional[str] = None
    metadata_to_render: Optional[str] = None
    video_type: VideoType
    file_path: str | None = Field(default=None)

    @property
    def resolution(self) -> Tuple[int, int]:
        return (self.resolution_x, self.resolution_y)


class Clip(UnvalidatedClip, table=True):
    id: UUID = Field(default_factory=uuid4, primary_key=True)
    # clip is child to both words AND a video
    # Set of words. Theoretically be from different utterances and transcripts. Consecutive words from same utterance will be cut together. in audio_video logic
    words: List[Word] = Relationship(back_populates="clips", link_model=WordClipLink)
    video_id: UUID = Field(default=None, foreign_key="video.id")
    video: Video = Relationship(back_populates="clips")
    created_at: datetime = Field(default=datetime.utcnow(), nullable=False)
    updated_at: datetime = Field(default_factory=datetime.utcnow, nullable=False)

In [9]:
# | export
T = TypeVar("T")
P = ParamSpec("P")


class PermissionDecorator(Protocol):
    def __call__(
        self, func: Callable[Concatenate[Any, P], T]
    ) -> Callable[Concatenate[Any, P], T]: ...


class DatabaseProtocol(Protocol):
    employee: Optional[Employee]

    def get_session_for_employee(self, public: bool = False) -> Session: ...
    def permissions_set_session_variables(
        self, session: Session, employee: Employee
    ) -> None: ...
    def permissions_clear_session_variables(self, session: Session) -> None: ...


DBType = TypeVar("DBType", bound=DatabaseProtocol)


def get_current_level(session: Session) -> str | None:
    current_level = session.execute(
        text("SELECT current_setting('app.current_permission_level')")
    ).scalar()
    return current_level


def permission_required(level: PermissionLevel):
    def decorator(
        func: Callable[Concatenate[DBType, Session, P], T],
    ) -> Callable[Concatenate[DBType, P], T]:
        """Decorator to require a certain permission level for a function. Unsets employee at the end of the function."""

        @wraps(func)
        def wrapper(self: DBType, *args: P.args, **kwargs: P.kwargs) -> T:
            if level == PermissionLevel.PUBLIC:
                with self.get_session_for_employee(public=True) as session:
                    return func(self, session, *args, **kwargs)
            with self.get_session_for_employee() as session:
                if not self.employee:
                    raise ValueError("Employee not provided")
                try:
                    # sets and then checks database permission levels at both an app-level and RLS level
                    self.permissions_set_session_variables(session, self.employee)
                    current_employee_permissions_level = get_current_level(session)
                    if current_employee_permissions_level is None:
                        raise PermissionError("Permissions not set for this session.")
                    if int(current_employee_permissions_level) < level.value:
                        raise PermissionError(
                            "Employee does not have sufficient permissions"
                        )
                    return func(self, session, *args, **kwargs)
                finally:
                    self.permissions_clear_session_variables(session)
                    self.employee = None

        wrapper._permission_level = level  # type: ignore - dynamic assignment that I'm not worrying about.
        return wrapper

    return decorator


def read(
    func: Callable[Concatenate[DBType, Session, P], T],
) -> Callable[Concatenate[DBType, P], T]:
    return permission_required(PermissionLevel.READ)(func)


def write(
    func: Callable[Concatenate[DBType, Session, P], T],
) -> Callable[Concatenate[DBType, P], T]:
    return permission_required(PermissionLevel.WRITE)(func)


def admin(
    func: Callable[Concatenate[DBType, Session, P], T],
) -> Callable[Concatenate[DBType, P], T]:
    return permission_required(PermissionLevel.ADMIN)(func)


def public(
    func: Callable[Concatenate[DBType, Session, P], T],
) -> Callable[Concatenate[DBType, P], T]:
    return permission_required(PermissionLevel.PUBLIC)(func)

In [10]:
# | export


class AbstractDatabase(ABC, DatabaseProtocol, Generic[DBType]):
    """Abstract base class for database operations related to users, files, transcriptions, and schemas.
    Use Unvalidated Models to save raw user input, validation occurs on the model save
    """

    def __init__(self):
        self.employee = None

    # OPERATIONS

    class Operations(ABC):
        @abstractmethod
        def create_all_tables(self) -> None:
            """Initializes the database."""
            raise NotImplementedError

        @abstractmethod
        def run_migrations(self) -> None:
            """Runs database migrations."""
            raise NotImplementedError

        @abstractmethod
        def close(self) -> None:
            """Closes the database connection."""
            raise NotImplementedError

        @abstractmethod
        def enable_row_level_security(self) -> None:
            """Enables row-level security for the database."""
            raise NotImplementedError

    @public
    @abstractmethod
    def set_session_variables(self, session: Session, employee: Employee) -> None:
        """Sets the current employee for the session."""
        raise NotImplementedError

    @abstractmethod
    def permissions_set_session_variables(
        self, session: Session, employee: Employee
    ) -> None:
        """Sets the current employee for the session. Requires a session to be passed in"""
        raise NotImplementedError

    @public
    @abstractmethod
    def clear_session_variables(self, session: Session) -> None:
        """Clears the session variables."""
        raise NotImplementedError

    @abstractmethod
    def permissions_clear_session_variables(self, session: Session) -> None:
        """Clears the session variables. Requires a session to be passed in"""
        raise NotImplementedError

    @abstractmethod
    def get_session_for_employee(self, public: bool = False) -> Session:
        """Returns a session for the database. Fails if an Auth'd User is not available unless public=True"""
        raise NotImplementedError

    # AUTH and PERMISSIONS

    def as_employee(self, employee: Employee) -> "AbstractDatabase[DBType]":
        """Returns an authenticated database authorized to act as the provided employee."""
        raise NotImplementedError

    @write
    @abstractmethod
    def update_employee(
        self,
        session: Session,
        employee_id: Union[str, UUID],
        values_to_update: dict[str, Any],
    ) -> Employee:
        """Updates an employee in the database."""
        raise NotImplementedError

    @write
    @abstractmethod
    def save_employee(
        self,
        session: Session,
        unvalidated_employee: CreateEmployee,
        company_id: UUID,
    ) -> Employee:
        """Creates an employee in the database."""
        raise NotImplementedError

    @public
    @abstractmethod
    def save_company_and_employee(
        self,
        session: Session,
        unvalidated_company: UnvalidatedCompany,
        unvalidated_employee: CreateEmployee,
    ) -> Tuple[Company, Employee]:
        """Creates a company and employee in the database. Intended for use with first-time signups"""
        raise NotImplementedError

    @public
    @abstractmethod
    def authenticate_employee(
        self, session: Session, email: str, password: str
    ) -> Optional[Employee]:
        """Authenticates an employee."""
        raise NotImplementedError

    @property
    @abstractmethod
    def operations(self) -> "Operations":
        """Property that should return an instance of Operations."""
        raise NotImplementedError

    # USERS

    @write
    @abstractmethod
    def save_user(self, session: Session, user: UnvalidatedUser) -> User:
        """Saves user information to the database."""
        raise NotImplementedError

    @write
    @abstractmethod
    def update_user(
        self, session: Session, user_id: UUID, values_to_update: dict[str, Any]
    ) -> User:
        """Updates user information in the database."""
        raise NotImplementedError

    @write
    def delete_user(self, session: Session, user_id: UUID) -> None:
        """Deletes a user from the database. Cascades to all transcriptions and file metadata"""
        raise NotImplementedError

    # FILES

    @write
    @abstractmethod
    def save_file_metadata(
        self, session: Session, metadata: UnvalidatedFileMetadata
    ) -> FileMetadata:
        """Saves metadata about a file uploaded by users."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_file_metadata(
        self, session: Session, file_id: Union[str, UUID]
    ) -> Optional[FileMetadata]:
        """Retrieves metadata about a file from the database."""
        raise NotImplementedError

    @write
    @abstractmethod
    def update_file_metadata(
        self, session: Session, file_id: UUID, values_to_update: dict[str, Any]
    ) -> FileMetadata:
        """Updates metadata about a file."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_file_metadata_from_list_of_transcript_ids(
        self, session: Session, transcript_ids: List[str]
    ) -> Sequence[FileMetadata]:
        """Retrieves metadata about a file from the database. Returns results in the order of the input list"""
        raise NotImplementedError

    # TRANSCRIPTIONS

    @write
    @abstractmethod
    def save_transcription(
        self,
        session: Session,
        transcription: UnvalidatedTranscription,
        utterances: List[UnvalidatedUtterance],
    ) -> Transcription:
        """Saves transcription data for a given file."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_transcription(
        self, session: Session, transcription_id: Union[str, UUID]
    ) -> Optional[Transcription]:
        """Retrieves a transcription from the database."""
        raise NotImplementedError

    @write
    @abstractmethod
    def update_transcription(
        self, session: Session, transcription_id: UUID, values_to_update: dict[str, Any]
    ) -> Transcription:
        """Updates a transcription in the database."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_transcriptions(
        self,
        session: Session,
        user_id: Union[str, UUID],
        transcription_ids: Optional[List[UUID]] = None,
    ) -> Sequence[Transcription]:
        """Retrieves all transcriptions and associated user information from the database."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_all_unique_transcriptions(
        self, session: Session, mode: Literal["file_name", "user_id"]
    ) -> Sequence[Transcription]:
        """Retrieves all unique transcriptions from the database."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_user(self, session: Session, user_id: Union[str, UUID]) -> Optional[User]:
        """Retrieves a user from the database."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_words_from_utterance_ids(
        self, session: Session, utterance_ids: List[UUID]
    ) -> Sequence[Word]:
        """Retrieves words from a list of utterance IDs."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_utterances(
        self, session: Session, utterance_ids: List[UUID]
    ) -> Sequence[Utterance]:
        """Retrieves utterances from a list of utterance IDs."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_transcriptions_by_user_ids(
        self, session: Session, user_ids: List[str]
    ) -> Sequence[Transcription]:
        """Retrieves transcriptions for multiple user IDs."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_latest_schema(self, session: Session) -> Optional[Schema]:
        """Retrieves the latest schema based on the created_at timestamp."""
        raise NotImplementedError

    # SCHEMAS

    @write
    @abstractmethod
    def save_schema(self, session: Session, schema: UnvalidatedSchema) -> Schema:
        """Saves a schema created from text to the database."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_schema(
        self, session: Session, schema_id: Union[str, UUID]
    ) -> Optional[Schema]:
        """Retrieves a schema from the database."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_all_users(self, session: Session) -> Sequence[User]:
        """Retrieves all users from the database."""
        raise NotImplementedError

    # CLIPS AND VIDEOS

    @write
    @abstractmethod
    def save_clip(
        self,
        session: Session,
        words: List[Word],
        clip: UnvalidatedClip,
        video_id: UUID | str,
    ) -> Clip:
        """Saves a clip to the database."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_clips_from_video_ids(
        self, session: Session, video_ids: List[UUID]
    ) -> Sequence[Clip]:
        """Retrieves clips from a list of video IDs."""
        raise NotImplementedError

    @write
    @abstractmethod
    def save_video(
        self, session: Session, video: Union[UnvalidatedVideo, Video]
    ) -> Video:
        """Saves a video to the database."""
        raise NotImplementedError

    @read
    @abstractmethod
    def get_video(
        self, session: Session, video_id: Union[str, UUID]
    ) -> Optional[Video]:
        """Retrieves a video from the database."""
        raise NotImplementedError

In [11]:
PermissionLevel["ADMIN"]

<PermissionLevel.ADMIN: 3>

In [12]:
import unittest


class MockDatabase:
    def __init__(self):
        self.current_permission_level = PermissionLevel.PUBLIC
        self.employee = None

    def get_session_for_employee(self, public=False):
        class MockSession:
            def __init__(self, db: "MockDatabase"):
                self.db = db

            def __enter__(self):
                return self

            def __exit__(self, *args):
                pass

            def execute(self, _):
                class MockScalar:
                    def __init__(self, db: "MockDatabase"):
                        self.db = db

                    def scalar(self):
                        return str(self.db.current_permission_level.value)

                return MockScalar(self.db)

        return MockSession(self)

    def as_employee(self, employee: Employee):
        self.employee = employee
        return self

    def permissions_set_session_variables(self, session: Session, employee: Employee):
        print(f"Setting session variables for employee: {employee}")
        pass

    def permissions_clear_session_variables(self, session: Session):
        print(f"Clearing session variables for session: {session}")
        pass

    @read
    def read_method(self, session: Session):
        print(f"Read method called with session: {session}")
        return "Read successful"

    @write
    def write_method(self, session: Session):
        print(f"Write method called with session: {session}")
        return "Write successful"

    @admin
    def admin_method(self, session: Session):
        print(f"Admin method called with session: {session}")
        return "Admin successful"

    @public
    def public_method(self, session: Session):
        print(f"Public method called with session: {session}")
        return "Public successful"


class TestPermissionDecorators(unittest.TestCase):
    def setUp(self):
        self.db = MockDatabase()
        self.employee = Employee(
            id=uuid4(),
            name="Test Employee",
            email="test@example.com",
            hashed_password="password".encode("utf-8"),
            company_id=uuid4(),
            permission_level=PermissionLevel.PUBLIC,
        )

    def test_public_method(self):
        self.assertEqual(
            self.db.as_employee(self.employee).public_method(), "Public successful"
        )

    def test_read_method(self):
        self.db.current_permission_level = PermissionLevel.READ
        self.assertEqual(
            self.db.as_employee(self.employee).read_method(), "Read successful"
        )

    def test_write_method(self):
        self.db.current_permission_level = PermissionLevel.WRITE
        self.assertEqual(
            self.db.as_employee(self.employee).write_method(), "Write successful"
        )

    def test_admin_method(self):
        self.db.current_permission_level = PermissionLevel.ADMIN
        self.assertEqual(
            self.db.as_employee(self.employee).admin_method(), "Admin successful"
        )

    def test_insufficient_permissions(self):
        self.db.current_permission_level = PermissionLevel.READ
        with self.assertRaises(
            PermissionError, msg="Writer not have sufficient permissions"
        ):
            self.db.as_employee(self.employee).write_method()
        with self.assertRaises(
            PermissionError, msg="Admin does not have sufficient permissions"
        ):
            self.db.as_employee(self.employee).admin_method()


def run_tests():
    suite = unittest.TestLoader().loadTestsFromTestCase(TestPermissionDecorators)
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)
    return result.wasSuccessful()


run_tests()

test_admin_method (__main__.TestPermissionDecorators.test_admin_method) ... ok
test_insufficient_permissions (__main__.TestPermissionDecorators.test_insufficient_permissions) ... ok
test_public_method (__main__.TestPermissionDecorators.test_public_method) ... ok
test_read_method (__main__.TestPermissionDecorators.test_read_method) ... ok
test_write_method (__main__.TestPermissionDecorators.test_write_method) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.015s

OK


Setting session variables for employee: email='test@example.com' name='Test Employee' permission_level=<PermissionLevel.PUBLIC: 0> id=UUID('f441dbc5-65c4-4410-88aa-3a3e64f380e1') company_id=UUID('09595da6-79ea-4a6a-bd66-4c4d9338348c') hashed_password=b'password' created_at=datetime.datetime(2024, 6, 28, 23, 27, 0, 787695) updated_at=datetime.datetime(2024, 6, 28, 23, 27, 0, 881718)
Admin method called with session: <__main__.MockDatabase.get_session_for_employee.<locals>.MockSession object at 0x10e416990>
Clearing session variables for session: <__main__.MockDatabase.get_session_for_employee.<locals>.MockSession object at 0x10e416990>
Setting session variables for employee: email='test@example.com' name='Test Employee' permission_level=<PermissionLevel.PUBLIC: 0> id=UUID('bec166ad-dec0-441d-8759-c65d7e7eb5fe') company_id=UUID('cfd1b6d2-ed04-475e-ac46-7f883d801b79') hashed_password=b'password' created_at=datetime.datetime(2024, 6, 28, 23, 27, 0, 787695) updated_at=datetime.datetime(2024

True

In [13]:
# | export
TModelOut = TypeVar("TModelOut", bound=SQLModel)


class SqlModelDatabase(AbstractDatabase["SqlModelDatabase"]):
    def __init__(self, database_url: str):
        self.engine = create_engine(database_url)
        SQLModel.metadata.create_all(self.engine)
        self.operations.enable_row_level_security()
        self.employee = None

    class Operations(AbstractDatabase.Operations):
        def __init__(self, database: "SqlModelDatabase"):
            self.database = database

        def create_all_tables(self) -> None:
            SQLModel.metadata.create_all(self.database.engine)

        def run_migrations(self) -> None:
            # Migrate company table
            self._migrate_table("company", [("name", "VARCHAR(255) NOT NULL")])

            # Migrate employee table
            self._migrate_table(
                "employee",
                [
                    ("name", "VARCHAR(255) NOT NULL"),
                    ("email", "VARCHAR(255) UNIQUE NOT NULL"),
                    ("hashed_password", "VARCHAR(255) NOT NULL"),
                    ("company_id", "UUID NOT NULL"),
                    ("permission_level", "VARCHAR(20) NOT NULL"),
                ],
            )

            # Add OrgBoundModel fields to relevant tables
            org_bound_fields = [
                ("company_id", "UUID NOT NULL"),
                ("created_by_id", "UUID NOT NULL"),
            ]

            for table in [
                "user",
                "file_metadata",
                "transcription",
                "word",
                "utterance",
                "schema",
                "clip",
            ]:
                self._migrate_table(table, org_bound_fields)

            # Migrate video table
            self._migrate_table(
                "video",
                [
                    ("resolution_x", "INTEGER"),
                    ("resolution_y", "INTEGER"),
                    ("fps", "INTEGER"),
                    ("render_status", "VARCHAR(20)"),
                    ("visibility", "VARCHAR(20) NOT NULL"),
                    *org_bound_fields,  # Include OrgBoundModel fields
                ],
            )

            # Migrate transcription table
            self._migrate_table(
                "transcription",
                [
                    ("embedded", "BOOLEAN DEFAULT FALSE"),
                    *org_bound_fields,  # Include OrgBoundModel fields
                ],
            )

            # Migrate clip table
            self._migrate_table(
                "clip",
                [
                    ("fps", "INTEGER"),
                    ("resolution_x", "INTEGER"),
                    ("resolution_y", "INTEGER"),
                    ("metadata_to_render", "VARCHAR(255) DEFAULT NULL"),
                    ("video_type", "VARCHAR(20) NOT NULL"),
                    *org_bound_fields,  # Include OrgBoundModel fields
                ],
            )
            self._drop_column_if_exists("clip", "text")

        def enable_row_level_security(self):
            with Session(self.database.engine) as session:
                for model in SQLModel.__subclasses__():
                    if hasattr(model, "__table__") and issubclass(model, OrgBoundModel):
                        print(f"Enabling RLS for {model.__tablename__}")
                        table_name = cast(str, model.__tablename__)  # type: ignore

                        # Check if RLS is already enabled
                        rls_enabled = session.execute(
                            text(f"""
                            SELECT relrowsecurity 
                            FROM pg_class 
                            WHERE relname = '{table_name}'
                        """)
                        ).scalar()

                        if not rls_enabled:
                            session.execute(
                                text(
                                    f"ALTER TABLE {table_name} ENABLE ROW LEVEL SECURITY"
                                )
                            )
                            print(f"Enabled RLS for table {table_name}")

                        # Check if policy already exists
                        policy_exists = session.execute(
                            text(f"""
                            SELECT 1 
                            FROM pg_policy 
                            WHERE polrelid = '{table_name}'::regclass
                        """)
                        ).scalar()

                        if not policy_exists:
                            if model == Video:
                                session.execute(
                                    text(f"""
                                    CREATE POLICY {table_name}_policy ON {table_name}
                                    USING (
                                        (company_id = current_setting('app.current_company_id')::uuid
                                        AND (
                                            current_setting('app.current_permission_level') = 'admin'
                                            OR (current_setting('app.current_permission_level') IN ('write', 'admin'))
                                            OR (current_setting('app.current_permission_level') = 'read' AND visibility != 'private')
                                            OR (created_by_id = current_setting('app.current_employee_id')::uuid)
                                        ))
                                        OR visibility = 'public'
                                    )
                                """)
                                )
                            else:
                                session.execute(
                                    text(f"""
                                    CREATE POLICY {table_name}_policy ON {table_name}
                                    USING (
                                        company_id = current_setting('app.current_company_id')::uuid
                                        AND (
                                            current_setting('app.current_permission_level') IN ('read', 'write', 'admin')
                                        )
                                    )
                                """)
                                )
                            print(f"Created RLS policy for table {table_name}")
                        else:
                            print(f"RLS policy already exists for table {table_name}")

                session.commit()
                print("Row-level security setup completed.")

        def _migrate_table(
            self, table_name: str, new_columns: List[Tuple[str, str]]
        ) -> None:
            from sqlalchemy import inspect, text

            inspector = inspect(self.database.engine)
            existing_columns = [
                column["name"] for column in inspector.get_columns(table_name)
            ]

            for column_name, column_type in new_columns:
                if column_name not in existing_columns:
                    with Session(self.database.engine) as session:
                        session.execute(
                            text(
                                f"ALTER TABLE {table_name} ADD COLUMN {column_name} {column_type}"
                            )
                        )
                        session.commit()
                        print(f"Added column {column_name} to {table_name}")
                else:
                    print(f"Column {column_name} already exists in {table_name}")

        def _drop_column_if_exists(self, table_name: str, column_name: str) -> None:
            from sqlalchemy import inspect, text

            inspector = inspect(self.database.engine)
            existing_columns = [
                column["name"] for column in inspector.get_columns(table_name)
            ]

            if column_name in existing_columns:
                with Session(self.database.engine) as session:
                    session.execute(
                        text(f"ALTER TABLE {table_name} DROP COLUMN {column_name}")
                    )
                    session.commit()
                    print(f"Dropped column {column_name} from {table_name}")
            else:
                print(f"Column {column_name} does not exist in {table_name}")

        def close(self) -> None:
            self.database.engine.dispose()

    def as_employee(self, employee: Employee):
        self.employee = employee
        return self

    def permissions_set_session_variables(self, session: Session, employee: Employee):
        """Same as set_session_variables, but without any kind of permissions. Use at your own risk."""
        session.execute(text(f"SET app.current_company_id = '{employee.company_id}'"))
        session.execute(
            text(f"SET app.current_permission_level = '{employee.permission_level}'")
        )
        session.flush()

    @public
    def set_session_variables(self, session: Session, employee: Employee):
        session.execute(text(f"SET app.current_company_id = '{employee.company_id}'"))
        session.execute(
            text(f"SET app.current_permission_level = '{employee.permission_level}'")
        )
        session.flush()

    @public
    def clear_session_variables(self, session: Session):
        session.execute(text("RESET app.current_company_id"))
        session.execute(text("RESET app.current_permission_level"))
        session.commit()

    def permissions_clear_session_variables(self, session: Session):
        session.execute(text("RESET app.current_company_id"))
        session.execute(text("RESET app.current_permission_level"))
        session.commit()

    def get_session_for_employee(self, public: bool = False) -> Session:
        if public:
            return Session(self.engine, expire_on_commit=False)
        if self.employee is None:
            raise PermissionError("Employee not set for this session.")
        return Session(self.engine, expire_on_commit=False)

    @public
    def authenticate_employee(
        self, session: Session, email: str, password: str
    ) -> Optional[Employee]:
        with Session(self.engine) as session:
            employee = session.exec(
                select(Employee).where(Employee.email == email)
            ).first()

            if employee and pbkdf2_sha256.verify(
                password, employee.hashed_password.decode("utf-8")
            ):
                return employee

        return None

    @write
    def save_employee(
        self,
        session: Session,
        unvalidated_employee: CreateEmployee,
        company_id: UUID,
    ) -> Employee:
        hashed_password = pbkdf2_sha256.hash(unvalidated_employee.password).encode(
            "utf-8"
        )
        company = session.exec(select(Company).where(Company.id == company_id)).first()
        if company is None:
            raise ValueError(f"Company with id {company_id} not found")
        print(unvalidated_employee.model_dump())
        employee_to_validate = Employee(
            **unvalidated_employee.model_dump(),
            hashed_password=hashed_password,
            company_id=company_id,
        )
        print("employee_to_validate.company_id", employee_to_validate.company_id)
        employee = Employee.model_validate(employee_to_validate)
        company.employees.append(employee)
        session.commit()
        session.refresh(employee)
        return employee

    @write
    def update_employee(
        self,
        session: Session,
        employee_id: str | UUID,
        values_to_update: dict[str, Any],
    ) -> Employee:
        employee = session.exec(
            select(Employee).where(Employee.id == employee_id)
        ).first()
        if employee is None:
            raise ValueError(f"Employee with id {employee_id} not found")
        for key, value in values_to_update.items():
            setattr(employee, key, value)
        session.add(employee)
        session.commit()
        session.refresh(employee)
        return employee

    @public
    def save_company_and_employee(
        self,
        session: Session,
        unvalidated_company: UnvalidatedCompany,
        unvalidated_employee: CreateEmployee,
    ) -> Tuple[Company, Employee]:
        company = Company.model_validate(unvalidated_company)
        session.add(company)
        session.commit()
        session.refresh(company)
        hashed_password = pbkdf2_sha256.hash(unvalidated_employee.password).encode(
            "utf-8"
        )
        employee_to_validate = Employee(
            **unvalidated_employee.model_dump(),
            hashed_password=hashed_password,
            company_id=company.id,
        )
        employee = Employee.model_validate(employee_to_validate)
        company.employees.append(employee)
        session.commit()
        session.refresh(employee)

        return company, employee

    def _save(
        self, session: Session, model: SQLModel, output_model: Type[TModelOut]
    ) -> TModelOut:
        valid_model = output_model.model_validate(model)
        session.add(valid_model)
        session.commit()
        session.refresh(valid_model)
        return valid_model

    @write
    def save_user(self, session: Session, user: UnvalidatedUser) -> User:
        return self._save(session, user, User)

    @write
    def delete_user(self, session: Session, user_id: UUID) -> None:
        with Session(self.engine) as session:
            user = session.exec(select(User).where(User.id == user_id)).first()
            if user is None:
                raise ValueError(f"User with id {user_id} not found")
            session.delete(user)
            session.commit()

    @write
    def update_user(
        self, session: Session, user_id: UUID, values_to_update: dict[str, Any]
    ) -> User:
        user = session.exec(select(User).where(User.id == user_id)).first()
        if user is None:
            raise ValueError(f"User with id {user_id} not found")
        for key, value in values_to_update.items():
            setattr(user, key, value)
        session.add(user)
        session.commit()
        session.refresh(user)
        return user

    @write
    def save_file_metadata(
        self, session: Session, metadata: UnvalidatedFileMetadata
    ) -> FileMetadata:
        return self._save(session, metadata, FileMetadata)

    @read
    def get_file_metadata(
        self, session: Session, file_id: Union[str, UUID]
    ) -> Optional[FileMetadata]:
        return session.exec(
            select(FileMetadata).where(FileMetadata.id == file_id)
        ).first()

    @write
    def update_file_metadata(
        self, session: Session, file_id: UUID, values_to_update: dict[str, Any]
    ) -> FileMetadata:
        metadata = session.exec(
            select(FileMetadata).where(FileMetadata.id == file_id)
        ).first()
        if metadata is None:
            raise ValueError(f"FileMetadata with id {file_id} not found")
        for key, value in values_to_update.items():
            setattr(metadata, key, value)
        session.add(metadata)
        session.commit()
        session.refresh(metadata)
        return metadata

    @read
    def get_file_metadata_from_list_of_transcript_ids(
        self, session: Session, transcript_ids: List[str]
    ) -> Sequence[FileMetadata]:
        order_case = case(
            {
                transcript_id: index
                for index, transcript_id in enumerate(transcript_ids)
            },
            value=Transcription.id,
        )
        return session.exec(
            select(FileMetadata)
            .options(joinedload(FileMetadata.transcription))  # type: ignore - selectinload type is weird
            .join(Transcription)
            .where(col(Transcription.id).in_(transcript_ids))
            .order_by(order_case)
        ).all()

    @write
    def save_transcription(
        self,
        session: Session,
        transcription: UnvalidatedTranscription,
        utterances: List[UnvalidatedUtterance],
    ) -> Transcription:
        if len(utterances) == 0:
            raise ValueError("At least one utterance must be provided")
        valid_transcription = Transcription.model_validate(transcription)
        session.add(valid_transcription)
        for utterance in utterances:
            words = [Word.model_validate(word) for word in utterance.words]
            utterance_to_save = Utterance.model_validate(utterance.model_copy(update={"words": words}))
            valid_transcription.utterances.append(
                utterance_to_save
            )
        session.commit()
        session.refresh(valid_transcription)
        # Eagerly load the associated utterances and words
        transcript_with_utterances_words = session.exec(
            select(Transcription)
            .options(joinedload(Transcription.utterances).joinedload(Utterance.words))  # type: ignore - selectinload type is weird
            .where(Transcription.id == valid_transcription.id)
        ).first()
        print("transcript_with_utterances_words", transcript_with_utterances_words)
        if transcript_with_utterances_words is None:
            raise ValueError(
                f"Transcription with id {valid_transcription.id} not found"
            )
        return transcript_with_utterances_words

    @read
    def get_transcription(
        self, session: Session, transcription_id: Union[str, UUID]
    ) -> Optional[Transcription]:
        return session.exec(
            select(Transcription).where(Transcription.id == transcription_id)
        ).first()

    @read
    def get_transcriptions(
        self,
        session: Session,
        user_id: Optional[Union[str, UUID]] = None,
        transcription_ids: Optional[List[UUID]] = None,
    ) -> Sequence[Transcription]:
        if transcription_ids:
            transcript_with_file_meta = (
                select(Transcription)
                .join(FileMetadata)
                .options(joinedload(Transcription.file_metadata))  # type: ignore - selectinload type is weird
                .where(col(Transcription.id).in_(transcription_ids))
            )
        else:
            transcript_with_file_meta = (
                select(Transcription)
                .join(FileMetadata)
                .options(joinedload(Transcription.file_metadata))  # type: ignore - selectinload type is weird
                .where(FileMetadata.user_id == user_id)
            )
        return session.exec(transcript_with_file_meta).all()

    @write
    def update_transcription(
        self, session: Session, transcription_id: UUID, values_to_update: dict[str, Any]
    ) -> Transcription:
        transcription = session.exec(
            select(Transcription).where(Transcription.id == transcription_id)
        ).first()
        if transcription is None:
            raise ValueError(f"Transcription with id {transcription_id} not found")
        for key, value in values_to_update.items():
            setattr(transcription, key, value)
        session.add(transcription)
        session.commit()
        session.refresh(transcription)
        return transcription

    @read
    def get_words_from_utterance_ids(
        self, session: Session, utterance_ids: List[UUID]
    ) -> Sequence[Word]:
        return session.exec(
            select(Word).where(col(Word.utterance_id).in_(utterance_ids))
        ).all()

    @read
    def get_utterances(
        self, session: Session, utterance_ids: List[UUID] | List[str]
    ) -> Sequence[Utterance]:
        return session.exec(
            select(Utterance)
            .where(col(Utterance.id).in_(utterance_ids))
            .options(selectinload(Utterance.words))  # type: ignore - selectinload type is weird
        ).all()

    @read
    def get_user(self, session: Session, user_id: str | UUID) -> Optional[User]:
        return session.exec(select(User).where(User.id == user_id)).first()

    @write
    def save_schema(self, session: Session, schema: UnvalidatedSchema) -> Schema:
        return self._save(session, schema, Schema)

    @read
    def get_schema(
        self, session: Session, schema_id: Union[str, UUID]
    ) -> Optional[Schema]:
        return session.exec(select(Schema).where(Schema.id == schema_id)).first()

    @read
    def get_transcriptions_by_user_ids(
        self, session: Session, user_ids: List[str]
    ) -> Sequence[Transcription]:
        transcript_with_file_meta = (
            select(Transcription)
            .options(selectinload(Transcription.utterances))  # type: ignore - selectinload type is weird
            .join(FileMetadata)
            .where(col(FileMetadata.user_id).in_(user_ids))
        )
        return session.exec(transcript_with_file_meta).all()

    @read
    def get_all_unique_transcriptions(
        self, session: Session, mode: Literal["file_name", "user_id"]
    ) -> Sequence[Transcription]:
        """
        Returns all unique transcriptions and utterances based on the mode specified.
        If mode is "file_name", returns all unique transcriptions based on the file name.
        If mode is "user_id", returns all unique transcriptions based on the user ID.
        """
        if mode == "file_name":
            unique_metadata_query = session.exec(
                select(FileMetadata).distinct(col(FileMetadata.file_name))
            )
            unique_metadata_ids = [result.id for result in unique_metadata_query]
            transcripts = session.exec(
                select(Transcription)
                .options(selectinload(Transcription.utterances))  # type: ignore - selectinload type is weird
                .options(
                    selectinload(Transcription.utterances).joinedload(Utterance.words)  # type: ignore - selectinload type is weird
                )
                .where(col(Transcription.file_id).in_(unique_metadata_ids))
            ).all()
            return transcripts
        elif mode == "user_id":
            raise NotImplementedError
        else:
            raise ValueError(f"Invalid mode: {mode}")

    @read
    def get_latest_schema(self, session: Session) -> Optional[Schema]:
        return session.exec(
            select(Schema).order_by(col(Schema.created_at).desc()).limit(1)
        ).first()

    @read
    def get_all_users(self, session: Session) -> Sequence[User]:
        return session.exec(select(User)).all()

    @write
    def save_clip(
        self,
        session: Session,
        words: List[Word],
        clip: UnvalidatedClip,
        video_id: UUID | str,
    ) -> Clip:
        """Saves a clip to the database."""
        unpacked_clip = clip.model_dump()
        valid_clip = Clip(**unpacked_clip)
        for word in words:
            valid_clip.words.append(word)
        video = session.exec(select(Video).where(Video.id == video_id)).first()
        if video is None:
            raise ValueError(f"Video with id {video_id} not found")
        video.clips.append(valid_clip)
        session.add(video)
        session.commit()
        session.refresh(video)
        session.refresh(valid_clip)
        clip_query = (
            select(Clip)
            .where(Clip.id == valid_clip.id)
            .options(selectinload(Clip.words))  # type: ignore - selectinload type is weird
        )
        # need to do this extra query to load the words eagerly
        final_clip = session.exec(clip_query).first()
        if final_clip is None:
            raise ValueError(f"Clip with id {valid_clip.id} not found")
        return final_clip

    @read
    def get_clips_from_video_ids(
        self, session: Session, video_ids: List[UUID]
    ) -> Sequence[Clip]:
        """Retrieves clips from a list of video IDs."""
        return session.exec(select(Clip).where(col(Clip.video_id).in_(video_ids))).all()

    @write
    def save_video(
        self, session: Session, video: Union[UnvalidatedVideo, Video]
    ) -> Video:
        """Saves a video to the database."""
        if isinstance(video, Video):
            # If it's an existing Video instance, merge it into the session
            merged_video = session.merge(video)
            session.commit()
            session.refresh(merged_video)
            return merged_video
        else:
            # If it's a new UnvalidatedVideo, validate and save it
            valid_video = Video.model_validate(video)
            session.add(valid_video)
            session.commit()
            session.refresh(valid_video)
            return valid_video

    @read
    def get_video(
        self, session: Session, video_id: Union[str, UUID]
    ) -> Optional[Video]:
        """Retrieves a video from the database."""
        return session.exec(select(Video).where(Video.id == video_id)).first()

    @property
    def operations(self) -> "Operations":
        return self.Operations(self)

In [15]:
from testcontainers.postgres import PostgresContainer  # type: ignore

# Assuming SqlModelDatabase and User are already defined
# Initialize the Postgres container
postgres = PostgresContainer("postgres:16.2")

# Create a test user instance

# Start the Postgres container and use the SqlModelDatabase API for operations
with postgres:
    database_url = cast(str, postgres.get_connection_url())  # type: ignore
    db = SqlModelDatabase(database_url=database_url)
    db.operations.create_all_tables()
    create_read_only = CreateEmployee(
        name="Max",
        email="max@example.com",
        password="password",
    )
    create_admin = CreateEmployee(
        name="Admin",
        email="admin@example.com",
        password="password",
        permission_level=PermissionLevel.ADMIN,
    )
    company = UnvalidatedCompany(name="Product Horse")
    company, admin_employee = db.save_company_and_employee(
        unvalidated_company=company, unvalidated_employee=create_admin
    )
    print(company.id)
    read_employee = db.as_employee(admin_employee).save_employee(
        unvalidated_employee=create_read_only, company_id=company.id
    )
    user_raw = UnvalidatedUser(
        name="Max",
        external_id="max@example.com",
        company_id=company.id,
        created_by_id=read_employee.id,
    )
    try:
        user = db.as_employee(read_employee).save_user(user_raw)
    except PermissionError as e:
        print(
            f"Permission error: {e}, now updating read only employee permissions with admin credential"
        )
        pass
    employee = db.as_employee(admin_employee).update_employee(
        read_employee.id, {"permission_level": PermissionLevel.ADMIN}
    )
    print(employee.permission_level)
    user = db.as_employee(employee).save_user(user_raw)
    metadata = UnvalidatedFileMetadata(
        user_id=user.id,
        file_name="test_file.txt",
        file_path="/path/to/test_file.txt",
        created_by_id=employee.id,
        company_id=company.id,
    )
    metadata = db.as_employee(employee).save_file_metadata(metadata)
    assert metadata.id is not None
    assert metadata.created_at is not None
    assert metadata.updated_at is not None
    schema = UnvalidatedSchema(
        input_text="", json_schema={}, company_id=company.id, created_by_id=employee.id
    )
    valid_schema = db.as_employee(employee).save_schema(schema)
    assert valid_schema.id is not None
    assert valid_schema.created_at is not None
    assert valid_schema.updated_at is not None
    assert db.as_employee(employee).get_schema(valid_schema.id) == valid_schema

    # Create a test user
    user_raw = UnvalidatedUser(
        name="John Doe",
        external_id="john.doe@example.com",
        company_id=company.id,
        created_by_id=employee.id,
    )
    user = db.as_employee(employee).save_user(user_raw)
    print(user)
    assert user.id is not None

    updated_user = db.as_employee(employee).update_user(user.id, {"name": "New Name"})
    assert updated_user.name == "New Name"

    # Create a test file metadata
    metadata_raw = UnvalidatedFileMetadata(
        user_id=user.id,
        file_name="test_file.txt",
        file_path="/path/to/test_file.txt",
        created_by_id=employee.id,
        company_id=company.id,
    )
    metadata = db.as_employee(employee).save_file_metadata(metadata_raw)

    # Create a test transcription with utterances and words
    transcription_raw = UnvalidatedTranscription(
        file_id=metadata.id,
        text="This is a test transcription.",
        created_by_id=employee.id,
        company_id=company.id,
    )

    utterance1 = UnvalidatedUtterance(
        confidence=0.95,
        end=1500,
        speaker="Speaker 1",
        start=0,
        text="This is a test",
        words=[
            UnvalidatedWord(
                text="This",
                start=0,
                end=400,
                confidence=0.98,
                speaker="Speaker 1",
                company_id=company.id,
                created_by_id=employee.id,
            ),
            UnvalidatedWord(
                text="is",
                start=401,
                end=600,
                confidence=0.97,
                speaker="Speaker 1",
                company_id=company.id,
                created_by_id=employee.id,
            ),
            UnvalidatedWord(
                text="a",
                start=601,
                end=800,
                confidence=0.96,
                speaker="Speaker 1",
                company_id=company.id,
                created_by_id=employee.id,
            ),
            UnvalidatedWord(
                text="test",
                start=801,
                end=1500,
                confidence=0.95,
                speaker="Speaker 1",
                company_id=company.id,
                created_by_id=employee.id,
            ),
        ],
        company_id=company.id,
        created_by_id=employee.id,
    )

    utterance2 = UnvalidatedUtterance(
        confidence=0.90,
        end=3000,
        speaker="Speaker 2",
        start=1501,
        text="transcription.",
        words=[
            UnvalidatedWord(
                text="transcription.",
                start=1501,
                end=3000,
                confidence=0.90,
                speaker="Speaker 2",
                company_id=company.id,
                created_by_id=employee.id,
            ),
        ],
        company_id=company.id,
        created_by_id=employee.id,
    )
    transcription = db.as_employee(employee).save_transcription(
        transcription_raw, [utterance1, utterance2]
    )

    # Validate the saved transcription
    assert transcription.id is not None
    assert transcription.created_at is not None
    assert transcription.updated_at is not None
    assert transcription.file_id == metadata.id
    assert transcription.text == "This is a test transcription."

    # Validate the saved utterances
    print(transcription.utterances)
    assert len(transcription.utterances) == 2

    utterance1_final = transcription.utterances[0]
    assert utterance1_final.confidence == 0.95
    assert utterance1_final.end == 1500
    assert utterance1_final.speaker == "Speaker 1"
    assert utterance1_final.start == 0
    assert utterance1_final.text == "This is a test"

    utterance2_final = transcription.utterances[1]
    assert utterance2_final.confidence == 0.90
    assert utterance2_final.end == 3000
    assert utterance2_final.speaker == "Speaker 2"
    assert utterance2_final.start == 1501
    assert utterance2_final.text == "transcription."

    # Validate the saved words
    assert len(utterance1_final.words) == 4
    assert utterance1_final.words[0].text == "This"
    assert utterance1_final.words[0].start == 0
    assert utterance1_final.words[0].end == 400
    assert utterance1_final.words[0].confidence == 0.98
    assert utterance1_final.words[0].speaker == "Speaker 1"

    assert len(utterance2_final.words) == 1
    assert utterance2_final.words[0].text == "transcription."
    assert utterance2_final.words[0].start == 1501
    assert utterance2_final.words[0].end == 3000
    assert utterance2_final.words[0].confidence == 0.90
    assert utterance2_final.words[0].speaker == "Speaker 2"

    user1_raw = UnvalidatedUser(
        name="User 1",
        external_id="user1@example.com",
        company_id=company.id,
        created_by_id=employee.id,
    )
    user1 = db.as_employee(employee).save_user(user1_raw)

    user2_raw = UnvalidatedUser(
        name="User 2",
        external_id="user2@example.com",
        company_id=company.id,
        created_by_id=employee.id,
    )
    user2 = db.as_employee(employee).save_user(user2_raw)

    metadata1 = UnvalidatedFileMetadata(
        user_id=user1.id,
        file_name="file1.txt",
        file_path="/path/to/file1.txt",
        company_id=company.id,
        created_by_id=employee.id,
    )
    metadata1 = db.as_employee(employee).save_file_metadata(metadata1)

    metadata2 = UnvalidatedFileMetadata(
        user_id=user2.id,
        file_name="file2.txt",
        file_path="/path/to/file2.txt",
        company_id=company.id,
        created_by_id=employee.id,
    )
    metadata2 = db.as_employee(employee).save_file_metadata(metadata2)

    transcription1 = UnvalidatedTranscription(
        file_id=metadata1.id,
        text="Transcription 1",
        company_id=company.id,
        created_by_id=employee.id,
    )
    transcription1 = db.as_employee(employee).save_transcription(
        transcription1, [utterance1, utterance2]
    )

    transcription2 = UnvalidatedTranscription(
        file_id=metadata2.id,
        text="Transcription 2",
        company_id=company.id,
        created_by_id=employee.id,
    )
    transcription2 = db.as_employee(employee).save_transcription(transcription2, [utterance1, utterance2])
    print(transcription2)

    user_ids = [str(user1.id), str(user2.id)]
    transcriptions = db.as_employee(employee).get_transcriptions_by_user_ids(user_ids)
    print(transcriptions[0].utterances)
    assert len(transcriptions) == 2
    assert transcriptions[0].file_id == metadata1.id
    assert transcriptions[1].file_id == metadata2.id
    assert transcriptions[0].utterances is not None
    assert transcriptions[1].utterances is not None

    transcriptions_from_list = db.as_employee(employee).get_transcriptions(
        transcription_ids=[transcription1.id, transcription2.id]
    )
    assert len(transcriptions_from_list) == 2
    assert transcriptions_from_list[0].id == transcription1.id
    assert transcriptions_from_list[1].id == transcription2.id
    assert transcriptions_from_list[0].file_metadata.id == metadata1.id
    assert transcriptions_from_list[1].file_metadata.id == metadata2.id

    updated_metadata = db.as_employee(employee).update_file_metadata(
        metadata1.id, {"file_name": "new_file_name.txt"}
    )
    assert updated_metadata.file_name == "new_file_name.txt"

    all_transcriptions: Sequence[Transcription] = db.as_employee(employee).get_all_unique_transcriptions(
        "file_name"
    )
    assert all_transcriptions[0]
    assert len(all_transcriptions[0].utterances) == 2

    transcript_ids = [str(transcription1.id), str(transcription2.id)]
    metadata = db.as_employee(employee).get_file_metadata_from_list_of_transcript_ids(transcript_ids)
    assert len(metadata) == 2
    assert metadata[0].id == metadata1.id
    assert metadata[1].id == metadata2.id
    assert metadata[0].transcription.id == transcription1.id
    assert metadata[1].transcription.id == transcription2.id
    # test order
    print(metadata[0].transcription.id)
    print(transcription1.id)
    assert metadata[0].transcription.id == transcription1.id
    assert metadata[1].transcription.id == transcription2.id

    updated_transcription = db.as_employee(employee).update_transcription(
        transcription1.id, {"embedded": True}
    )
    get_transcription = db.as_employee(employee).get_transcription(transcription1.id)
    assert get_transcription is not None
    assert get_transcription.embedded is True

    print(utterance1_final.id)
    print(utterance2_final.id)
    get_words = db.as_employee(employee).get_words_from_utterance_ids(
        [utterance1_final.id, utterance2_final.id]
    )
    print(get_words)
    print(len(get_words))
    assert len(get_words) == 5

    db.as_employee(employee).delete_user(user1.id)
    assert db.as_employee(employee).get_user(user1.id) is None
    assert db.as_employee(employee).get_file_metadata(metadata1.id) is None

    video = UnvalidatedVideo(
        duration=10, title="test", company_id=company.id, created_by_id=employee.id
    )
    video = db.as_employee(employee).save_video(video)
    assert video.id is not None
    # update video
    video.render_status = RenderStatus.complete
    video = db.as_employee(employee).save_video(video)
    print(video)
    clip_a = UnvalidatedClip(
        duration=10,
        text="test",
        utterance_start=0,
        utterance_end=1000,
        fps=24,
        resolution_x=1280,
        resolution_y=720,
        video_type=VideoType.video,
        company_id=company.id,
        created_by_id=employee.id,
    )
    clip_2 = UnvalidatedClip(
        duration=10,
        utterance_start=1000,
        utterance_end=2000,
        fps=24,
        resolution_x=1280,
        resolution_y=720,
        video_type=VideoType.video,
        company_id=company.id,
        created_by_id=employee.id,
    )
    print(utterance1.words)
    clip: Clip = db.as_employee(employee).save_clip(utterance1_final.words, clip_a, video.id)
    assert clip.id is not None
    assert clip.words == utterance1_final.words
    got_clips = db.as_employee(employee).get_clips_from_video_ids([video.id])
    assert len(got_clips) == 1
    assert got_clips[0].id == clip.id

    all_utterances = db.as_employee(employee).get_utterances([utterance1_final.id, utterance2_final.id])
    assert len(all_utterances) == 2
    assert len(all_utterances[0].words) > 1
    assert len(all_utterances[1].words) == 1
    assert all_utterances[0].id == utterance1_final.id
    assert all_utterances[1].id == utterance2_final.id

Pulling image postgres:16.2
Container started: a4df749b8332
Waiting to be ready...
Waiting to be ready...
Waiting to be ready...
Waiting to be ready...


Row-level security setup completed.
bfd97fdf-2646-4dac-9e6f-9d8c4b862d59
{'email': 'max@example.com', 'name': 'Max', 'permission_level': <PermissionLevel.READ: 1>, 'password': 'password'}
employee_to_validate.company_id bfd97fdf-2646-4dac-9e6f-9d8c4b862d59
Permission error: Employee does not have sufficient permissions, now updating read only employee permissions with admin credential
3
created_by_id=UUID('9bbdb9b4-0629-42a6-b973-0984409f06e6') name='John Doe' external_id='john.doe@example.com' created_at=datetime.datetime(2024, 6, 28, 23, 27, 0, 803605) company_id=UUID('bfd97fdf-2646-4dac-9e6f-9d8c4b862d59') id=UUID('788dd289-9afd-4a0d-b86f-4e5bde16fcba') updated_at=datetime.datetime(2024, 6, 28, 23, 27, 32, 673656)
transcript_with_utterances_words created_by_id=UUID('9bbdb9b4-0629-42a6-b973-0984409f06e6') file_id=UUID('dae8a3f2-fdc7-4b59-bd9f-f49f79548f83') id=UUID('c9c76f03-2ddc-40da-ae73-fe3d92c0c285') updated_at=datetime.datetime(2024, 6, 28, 23, 27, 32, 686900) company_id=UUID('b

In [ ]:
PG_URL = "postgresql://localhost:5432/product_horse"
db = SqlModelDatabase(database_url=PG_URL)
db.operations.run_migrations()

Column embedded already exists in transcription
Column resolution_x already exists in video
Column resolution_y already exists in video
Column fps already exists in video
Column render_status already exists in video
Column fps already exists in clip
Column resolution_x already exists in clip
Column resolution_y already exists in clip
Column metadata_to_render already exists in clip
Column video_type already exists in clip
Dropped column text from clip


note to self for morning -- something is awry with the UnvalidatedWord vs Word usage -- probably needs ot be flipped

# TODO: need to update the below tests to get whole-system tests running again. Missing lots of methods + the video and clip objects

In [ ]:
# from product_horse.core import run_test_case_with_pdb
# from hypothesis import strategies as st
# from hypothesis.stateful import (
#     RuleBasedStateMachine,
#     rule,
#     Bundle,
#     invariant,
#     precondition,
#     consumes,
# )


# def utterance_strategy():
#     return st.builds(
#         Utterance,
#         confidence=st.floats(min_value=0.0, max_value=1.0),
#         end=st.integers(min_value=0, max_value=922337203685477),
#         speaker=st.text(),
#         start=st.integers(min_value=0, max_value=922337203685477),
#         text=st.text(),
#     )


# class DatabaseStateMachine(RuleBasedStateMachine):
#     _is_setup_done = False  # Class-level flag to ensure setup is done only once

#     @classmethod
#     def setup_class(cls):
#         if not cls._is_setup_done:
#             # Initialize the Postgres container
#             cls.postgres_container = PostgresContainer("postgres:16.2")
#             cls.postgres_container.start()
#             database_url = cls.postgres_container.get_connection_url()
#             cls.database = SqlModelDatabase(database_url=database_url)
#             # Ensure tables are created
#             cls._is_setup_done = True

#     def __init__(self):
#         super().__init__()
#         self.setup_class()
#         self.user_count = 0
#         self.file_metadata_count = 0
#         self.transcription_count = 0
#         self.schema_count = 0

#     users = Bundle("users")
#     file_metadatas = Bundle("file_metadatas")
#     transcriptions = Bundle("transcriptions")

#     @rule(target=users, name=st.text(), email=st.text())
#     def create_user(self, name: str, email: str):
#         raw_user = UnvalidatedUser(name=name, external_id=email)
#         user = self.database.save_user(raw_user)
#         self.user_count += 1
#         return user

#     @rule(
#         target=file_metadatas,
#         user=consumes(users),
#         file_name=st.text(),
#         file_path=st.text(),
#     )
#     @precondition(lambda self: self.user_count > 0)
#     def create_file_metadata(self, user: User, file_name: str, file_path: str):
#         raw_metadata = UnvalidatedFileMetadata(
#             user_id=user.id, file_name=file_name, file_path=file_path
#         )
#         metadata = self.database.save_file_metadata(raw_metadata)
#         assert metadata.id is not None
#         assert metadata.created_at is not None
#         assert metadata.updated_at is not None
#         self.file_metadata_count += 1
#         return metadata

#     @rule(
#         target=transcriptions,
#         file_metadata=consumes(file_metadatas),
#         text=st.text(),
#         utterances=st.lists(utterance_strategy()),
#     )
#     @precondition(lambda self: self.file_metadata_count > 0)
#     def create_transcription(
#         self, file_metadata: FileMetadata, text: str, utterances: List[Utterance]
#     ):
#         raw_transcription = UnvalidatedTranscription(
#             file_id=file_metadata.id, text=text
#         )
#         transcription = self.database.save_transcription(raw_transcription, utterances)
#         assert transcription.id is not None
#         assert transcription.created_at is not None
#         assert transcription.updated_at is not None
#         self.transcription_count += 1
#         return transcription

#     @rule(schema_text=st.text())
#     def create_schema(self, schema_text: str):
#         raw_schema = UnvalidatedSchema(input_text=schema_text, json_schema={})
#         schema = self.database.save_schema(raw_schema)
#         assert schema.id is not None
#         assert schema.created_at is not None
#         assert schema.updated_at is not None
#         self.schema_count += 1

#     @rule(users=st.lists(consumes(users)))
#     @precondition(lambda self: self.user_count > 0 and self.transcription_count > 0)
#     def test_get_transcriptions_by_user_ids(self, users: List[User]):
#         user_ids = [str(user.id) for user in users]
#         transcriptions = self.database.get_transcriptions_by_user_ids(user_ids)
#         for transcription in transcriptions:
#             assert transcription.file_id is not None
#             file_metadata = self.database.get_file_metadata(transcription.file_id)
#             assert file_metadata is not None
#             assert file_metadata.user_id in user_ids

# # not testing this for now
#     # @rule(transcript_ids=st.lists(st.text()))
#     # def test_get_file_metadata_from_list_of_transcript_ids(self, transcript_ids: List[str]):
#     #     metadata = self.database.get_file_metadata_from_list_of_transcript_ids(transcript_ids)
#     #     assert len(metadata) == len(transcript_ids)

#     @rule()
#     @precondition(lambda self: self.schema_count > 0)
#     def test_get_latest_schema(self):
#         latest_schema = self.database.get_latest_schema()
#         assert latest_schema is not None

#     @invariant()
#     def user_count_is_correct(self):
#         assert self.user_count >= 0

#     @invariant()
#     def file_metadata_count_is_correct(self):
#         assert self.file_metadata_count >= 0

#     @invariant()
#     def transcription_count_is_correct(self):
#         assert self.transcription_count >= 0

#     @invariant()
#     def schema_count_is_correct(self):
#         assert self.schema_count >= 0


# TestDatabase = DatabaseStateMachine.TestCase
# run_test_case_with_pdb(TestDatabase, pdb_flag=False)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore

### Experimental

I should refactor this to be an API like:

`db.read(Transcription).on({'id': [transcript_ids]})`

And move the underlying code to pure sqlalchemy2

Instance - CRUD - fields (with: dict that has fields -- if it's a list of ids it filters on that, if it's a lambda it uses that as a filter.) - order by ({field: 'desc/asc'}) - optional modifier (one, all, first, limit)

From nothing other than plain python classes, auto-generate a statically-typed ORM.

Potential features:
- Auto-generate all relationship columns with eager loading and pagination -- no need to remember back_populates or implementing yield_per
- Handles all session management so you don't need to think about sqlalchemy at all
- basic auth functions and row-level security
- Handles relationship length validation (use event dispatch system)
- automatically adds UUID indices, created_at, updated_at
- Auto-types the fields by generating CRUD pydantic models like `CreateTranscription`, `ReadTranscription`, etc.
-- API that automatically covers most common cases that people will hit when using attributes
- Handles migrations (through atlas)


This is all written statically to a file in your codebase following https://peps.python.org/pep-0484/ so you can inspect the implementation at any time.

Maybe I can get the name db https://github.com/lgastako/db from this guy.

First: generate the models necessary from the original models given.

Initial sketch below

In [ ]:
from typing import Type, Dict, Any, Optional, Tuple
from sqlmodel import SQLModel, Field, Relationship
from pydantic import BaseModel, create_model
from datetime import datetime
from uuid import UUID
from rich.console import Console

console = Console()


def generate_sub_models(table_class: Type[SQLModel]) -> Dict[str, Type[BaseModel]]:
    fields = table_class.__fields__
    sub_models = {}

    def is_relationship_field(field: Any) -> bool:
        return field.annotation == Relationship

    def is_timing_field(field_name: str) -> bool:
        return field_name in ["created_at", "updated_at"]

    def is_id_field(field_name: str) -> bool:
        return field_name == "id"

    # Create model for creating new records
    create_fields = {
        name: (field.annotation, ...)
        for name, field in fields.items()
        if not (
            is_id_field(name) or is_relationship_field(field) or is_timing_field(name)
        )
    }
    CreateModel = create_model(f"Create{table_class.__name__}", **create_fields)
    sub_models["create"] = CreateModel

    # Create model for updating records
    update_fields = {
        name: (field.annotation, None)
        for name, field in fields.items()
        if not is_id_field(name)
    }
    UpdateModel = create_model(f"Update{table_class.__name__}", **update_fields)
    sub_models["update"] = UpdateModel

    # Create model for reading records
    read_fields = {name: (field.annotation, ...) for name, field in fields.items()}
    ReadModel = create_model(f"Read{table_class.__name__}", **read_fields)
    sub_models["read"] = ReadModel

    return sub_models


# Example usage
sub_models = generate_sub_models(User)
CreateUser = sub_models["create"]
UpdateUser = sub_models["update"]
ReadUser = sub_models["read"]

console.print("CreateUser.schema()", CreateUser.schema())
console.print("UpdateUser.schema()", UpdateUser.schema())
console.print("ReadUser.schema()", ReadUser.schema())

CreateUser.schema()
{
    'properties': {
        'name': {'title': 'Name', 'type': 'string'},
        'external_id': {'title': 'External Id', 'type': 'string'}
    },
    'required': ['name', 'external_id'],
    'title': 'CreateUser',
    'type': 'object'
}

UpdateUser.schema()
{
    'properties': {
        'name': {'default': None, 'title': 'Name', 'type': 'string'},
        'external_id': {'default': None, 'title': 'External Id', 'type': 'string'},
        'created_at': {'default': None, 'format': 'date-time', 'title': 'Created At', 'type': 'string'},
        'updated_at': {'default': None, 'format': 'date-time', 'title': 'Updated At', 'type': 'string'}
    },
    'title': 'UpdateUser',
    'type': 'object'
}

ReadUser.schema()
{
    'properties': {
        'name': {'title': 'Name', 'type': 'string'},
        'external_id': {'title': 'External Id', 'type': 'string'},
        'id': {'format': 'uuid', 'title': 'Id', 'type': 'string'},
        'created_at': {'format': 'date-time', 'title': 'Created At', 'type': 'string'},
        'updated_at': {'format': 'date-time', 'title': 'Updated At', 'type': 'string'}
    },
    'required': ['name', 'external_id', 'id', 'created_at', 'updated_at'],
    'title': 'ReadUser',
    'type': 'object'
}

Next, generate the code in a python file.

Below is a simple approach, but better to use something like Jinja and write the entire implementation, not just a stub

In [ ]:
from typing import Type, Dict, Any, Optional, Set
from sqlmodel import SQLModel, Field, Relationship
from pydantic import BaseModel, create_model
from datetime import datetime
from uuid import UUID


def write_models_to_stub(
    sub_models: Dict[str, Type[BaseModel]], filename: str = "model_stubs.pyi"
):
    import_lines: Set[str] = set()
    class_definitions = []

    # Collect import lines and class definitions
    for model_name, model in sub_models.items():
        model_name = model.__name__  # Ensure proper class name capitalization
        base_class_name = "BaseModel" if issubclass(model, BaseModel) else "SQLModel"
        class_definition = f"class {model_name}({base_class_name}):\n"
        for field_name, field_type in model.__annotations__.items():
            type_name = _get_type_name(field_type)
            module_name = _get_module_name(field_type)
            if module_name and module_name != "builtins":
                import_lines.add(f"from {module_name} import {type_name}\n")
            class_definition += f"    {field_name}: {type_name}\n"
        class_definition += "\n"
        class_definitions.append(class_definition)

    # Write to file
    with open(filename, "w") as f:
        f.writelines(sorted(import_lines))
        f.write("from pydantic import BaseModel\n")
        f.write("from sqlmodel import SQLModel\n")
        f.write("\n")
        f.writelines(class_definitions)


def _get_type_name(t: Any) -> str:
    if hasattr(t, "__name__"):
        return t.__name__
    elif hasattr(t, "_name"):
        return t._name
    elif hasattr(t, "__origin__"):
        base = t.__origin__.__name__
        args = ", ".join(_get_type_name(arg) for arg in t.__args__)
        return f"{base}[{args}]"
    else:
        return str(t)


def _get_module_name(t: Any) -> Optional[str]:
    if hasattr(t, "__module__"):
        return t.__module__
    elif hasattr(t, "__origin__"):
        origin = t.__origin__
        if hasattr(origin, "__module__"):
            return origin.__module__
    return None


# Example usage
sub_models = generate_sub_models(User)
write_models_to_stub(sub_models)

Here's the code generated. (well, some of it's generated right now. But all would be ideally). Approach is to auto-create all of this in a separate python package and import it in.

May be able to do with ParamSpec?

1. TypeVar T: This captures the return type of the function.
2. ParamSpec P: This captures all parameters of the function except for self. It allows us to preserve the exact signature of the method being decorated.
3. Concatenate[Any, P]: This is used to add the self parameter to the function signature. The Any represents self, and P represents all other parameters.
4. Callable[Concatenate[Any, P], T]: This represents a method that takes self and all the parameters captured by P, and returns a value of type T.
By using these advanced typing features, we tell the type ch

In [ ]:
from typing import Generic, overload, TypeVar, Type, Any, cast
from sqlmodel import SQLModel
from datetime import datetime
from uuid import UUID
from pydantic import BaseModel


class CreateUser(BaseModel):
    name: str
    external_id: str


class UpdateUser(BaseModel):
    name: str
    external_id: str
    created_at: datetime
    updated_at: datetime


class ReadUser(BaseModel):
    name: str
    external_id: str
    id: UUID
    created_at: datetime
    updated_at: datetime


T = TypeVar("T", bound=SQLModel)


class CreateHelper(Generic[T]):
    def __init__(self, model_class: Type[T]):
        self.model_class = model_class

    @overload
    def on(self, name: str, external_id: str) -> T: ...

    @overload
    def on(self, **kwargs: Any) -> T: ...

    def on(self, *args: Any, **kwargs: Any) -> T:
        return self.model_class(**kwargs)


class CreateUserHelper(CreateHelper[User]):
    @overload
    def on(self, name: str, external_id: str) -> User: ...

    @overload
    def on(self, **kwargs: Any) -> User: ...

    def on(self, *args: Any, **kwargs: Any) -> User:
        if args:
            raise TypeError("This method does not accept positional arguments")
        return self.model_class(**kwargs)


class Database:
    def create(self, model_class: Type[T]) -> CreateHelper[T]:
        if model_class == User:
            return cast(CreateHelper[T], CreateUserHelper(User))
        else:
            return CreateHelper(model_class)

NameError: name 'User' is not defined

In [ ]:
db = Database()
user = db.create(User).on(name="Max", external_id="max@example.com")
print(user)

name='Max' external_id='max@example.com' id=UUID('c8eb2137-950f-49b0-8da9-dd59c43a945d') created_at=datetime.datetime(2024, 6, 17, 2, 5, 50, 832953) updated_at=datetime.datetime(2024, 6, 17, 2, 5, 54, 153971)
